In [29]:
import pickle
import os
import numpy as np
import pandas as pd

from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.models.ctm import CombinedTM

In [30]:
# obligé de reprendre les données ici car bert n'a pas été entrainé sur dataset_final.pkl, donc j'avais un problème de matching entre les embeddings et les textes
df = pd.read_csv("../data/preprocessed/reviews_trust_clean.csv")
df_sans_contexte = pd.read_csv("../data/preprocessed/reviews_trust_clean_stopwords_supprimer.csv")
texts_sans_contexte = df_sans_contexte["clean_comment"].astype(str).tolist()
texts_avec_contexte = df["clean_comment"]

with open("../data/embeddings/emb_sbert_fr.pkl", "rb") as f:
    X_ctx = pickle.load(f)
    
X_ctx = np.asarray(X_ctx)
assert X_ctx.shape[0] == len(texts_sans_contexte), "Mismatch nb docs vs nb embeddings"

In [31]:
tp = TopicModelDataPreparation("camembert-base")

training_dataset = tp.fit(
    text_for_contextual=texts_avec_contexte,
    text_for_bow=texts_sans_contexte,
    custom_embeddings=X_ctx
)

bow_size = len(tp.vocab)
ctx_size = X_ctx.shape[1]
bow_size, ctx_size

(20726, 768)

In [32]:
K = 15 # nombre de topics
ctm = CombinedTM(
    bow_size=bow_size,
    contextual_size=ctx_size,
    n_components=K,
    num_epochs=20,
    num_data_loader_workers=0
)

ctm.fit(training_dataset) # entraînement

Epoch: [20/20]	 Seen Samples: [300800/301800]	Train Loss: 230.46601880661984	Time: 0:00:03.837825: : 20it [01:17,  3.85s/it]
100%|██████████| 236/236 [00:02<00:00, 94.14it/s]


In [33]:
topics_words = ctm.get_topic_lists(20)  #  le nombre de mots à afficher par topic
for k, words in enumerate(topics_words):
    print(f"Topic {k}: {', '.join(words)}")

Topic 0: fuir, éviter, voleurs, honte, nul, voyage, appels, arnaque, pire, mails, fuyez, commandes, malgré, impossible, déconseille, sav, incompétent, inexistant, font, réponses
Topic 1: est, ai, site, plus, faire, vente, si, commande, qu, bien, mois, jamais, service, car, privée, achat, après, rien, non, fait
Topic 2: fassent, expédiées, beaux, financière, agréables, bel, willemse, abouti, beau, écrite, logiciel, permis, sanitaire, récupérée, pensent, myphotobook, fins, belle, retient, servira
Topic 3: description, rapide, emballé, correspondant, conforme, satisfaite, super, recommande, excellent, conformes, facile, prévu, tôt, prévue, descriptif, rapport, parfaitement, emballés, bonne, respectés
Topic 4: colis, commande, jours, mail, après, remboursement, ai, service, rien, être, avoir, qu, mois, livraison, fois, donc, plus, est, client, toujours
Topic 5: taille, petit, robe, grand, couleur, trop, robes, shirt, petite, 38, photo, tailles, noir, correspond, tissu, blanc, tee, dommage,

In [34]:
doc_topic = ctm.get_doc_topic_distribution(training_dataset)  # shape (n_docs, K)
df["topic_id"] = np.argmax(doc_topic, axis=1)
df["topic_confidence"] = doc_topic.max(axis=1)

100%|██████████| 236/236 [00:02<00:00, 93.62it/s]


In [ ]:
os.makedirs("./artifacts/ctm", exist_ok=True)
os.makedirs("./artifacts/ctm/exports", exist_ok=True)

# ctm.save(models_dir="./artifacts/ctm/model")

# with open("./artifacts/ctm/vocab.pkl", "wb") as f:
#     pickle.dump(tp.vocab, f)

pd.DataFrame({
    "topic_id": np.arange(len(topics_words)),
    "top_words": [", ".join(w) for w in topics_words]
}).to_csv("./artifacts/ctm/exports/topics_top_words.csv", index=False)

np.save("./artifacts/ctm/exports/doc_topic.npy", doc_topic)

df.to_csv("./artifacts/ctm/exports/reviews_with_topics.csv", index=False)

c:\IA\trustpilot\venv\Lib\site-packages\contextualized_topic_models\models\ctm.py:640: Warning: This is an experimental feature that we has not been fully tested. Refer to the following issue:https://github.com/MilaNLProc/contextualized-topic-models/issues/38
  warnings.warn(


In [36]:
topic_counts = df["topic_id"].value_counts().sort_index()
display(topic_counts)

topic_id
0     1567
1      314
2     1017
3     1364
4      386
5      954
6      433
7     1211
8     1234
9     1685
10     895
11    1028
12    1269
13     789
14     944
Name: count, dtype: int64

In [37]:
def examples_for_topic(t, n=5):
    return df[df["topic_id"]==t]["clean_comment"].head(n).tolist()

for t in range(K):
    ex = examples_for_topic(t, 3)
    print(f"\n=== Topic {t} ===")
    for e in ex:
        print("-", e[:200])


=== Topic 0 ===
- commande téléphone etat a+ . livraison d un vieux téléphone pourri sans batterie rayé partout et inaudible ! ! ! super l affaire 300 euros a la poubelle merci showroomprivee ! ! ! passez votre chemin 
- commande passée pour une vente lacoste , livraison 15 nous après la date prévue , déjà 6 semaines après l'achat , sur 3 produits , 2 manquants . pas pro , pas sérieux , aucun geste commercial hormis l
- annulation de commande après 2 mois d ’ attente dans un geste et sans explication . retard de livraison et report à 3 reprise pour se résultat incompréhensible et inadmissible je recommande pas le sit

=== Topic 1 ===
- j'ai toujours été cliente chez showroom , jamais déçue . il y a quelques jours , je reçois un article non conforme . on m'envoie le bon retour . je dois faire l'avance des frais , hors de question , o
- a fuir , j'ai acheté un matelas a 400balles soit disant à moins de 70 % que chez le fabricant et livraison payante 20 eur.aparament ils savaient pas que